In [1]:
%pip install --upgrade "jax[cuda12]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 60.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 MB 21.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 47.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82
  Attempting uninstall: jax-cuda12-plugin
    Found existing installation: jax-cuda12-plugin 0.7.2
    Uninstalling jax-cuda12-plugin-0.7.2:
      Succes

In [4]:
!pip install --upgrade optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.0/403.0 kB 7.3 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: optax
    Found existing installation: optax 0.2.7
    Uninstalling optax-0.2.7:
      Successfully uninstalled optax-0.2.7


In [5]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
from tqdm import tqdm
from pathlib import Path
import urllib.request

In [6]:
print(jax.devices())

[CudaDevice(id=0), CudaDevice(id=1)]


In [7]:
data_path = Path("data")
data_path.mkdir(exist_ok=True)
file_path = data_path / "input.txt"
if not file_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, file_path)

In [8]:
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

In [10]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

In [11]:
data = jnp.array(encode(text), dtype=jnp.int32)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 256
batch_size = 128
emb_dim = 256
num_heads = 1
num_layers = 8
dropout_rate = 0.1
max_lr = 3e-4
weight_decay = 0.01
total_steps = 1000
eval_iters = 100

In [13]:
class CausalSelfAttention(nn.Module):
    emb_dim: int
    num_heads: int
    dropout_rate: float

    def setup(self):
        assert self.emb_dim % self.num_heads == 0
        self.head_dim = self.emb_dim // self.num_heads
        self.qkv = nn.Dense(self.emb_dim * 3, use_bias=False)
        self.proj = nn.Dense(self.emb_dim)
        self.dropout = nn.Dropout(rate=self.dropout_rate)

    def __call__(self, x, deterministic=True):
        B, T, _ = x.shape
        qkv = self.qkv(x)  # (B, T, 3 * emb_dim)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape to (B, num_heads, T, head_dim)
        q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        # Scaled dot‑product attention
        attn_weights = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)

        # mask: -1e4 is safe for float16 (does not overflow to -inf)
        causal_mask = jnp.tril(jnp.ones((T, T))).reshape(1, 1, T, T)
        attn_weights = jnp.where(causal_mask == 0, -1e4, attn_weights)

        attn_weights = nn.softmax(attn_weights, axis=-1)
        attn_weights = self.dropout(attn_weights, deterministic=deterministic)

        out = jnp.matmul(attn_weights, v)                 # (B, num_heads, T, head_dim)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, -1) # (B, T, emb_dim)
        out = self.proj(out)
        out = self.dropout(out, deterministic=deterministic)
        return out

In [14]:
class MLP(nn.Module):
    emb_dim: int
    dropout_rate: float

    def setup(self):
        self.fc1 = nn.Dense(4 * self.emb_dim)
        self.fc2 = nn.Dense(self.emb_dim)
        self.dropout = nn.Dropout(rate=self.dropout_rate)

    def __call__(self, x, deterministic=True):
        x = nn.gelu(self.fc1(x))
        x = self.dropout(x, deterministic=deterministic)
        x = self.fc2(x)
        return x


In [15]:
class DecoderBlock(nn.Module):
    emb_dim: int
    num_heads: int
    dropout_rate: float

    def setup(self):
        self.ln1 = nn.LayerNorm()
        self.attn = CausalSelfAttention(self.emb_dim, self.num_heads, self.dropout_rate)
        self.ln2 = nn.LayerNorm()
        self.mlp = MLP(self.emb_dim, self.dropout_rate)

    def __call__(self, x, deterministic=True):
        x = x + self.attn(self.ln1(x), deterministic=deterministic)
        x = x + self.mlp(self.ln2(x), deterministic=deterministic)
        return x

In [16]:
class GPT(nn.Module):
    vocab_size: int
    emb_dim: int
    block_size: int
    num_heads: int
    num_layers: int
    dropout_rate: float

    def setup(self):
        self.token_embed = nn.Embed(self.vocab_size, self.emb_dim)
        self.pos_embed = self.param(
            "pos_embed", nn.initializers.normal(0.02), (1, self.block_size, self.emb_dim)
        )
        self.dropout_emb = nn.Dropout(rate=self.dropout_rate)
        self.layers = [
            DecoderBlock(self.emb_dim, self.num_heads, self.dropout_rate)
            for _ in range(self.num_layers)
        ]
        self.ln_f = nn.LayerNorm()
        self.lm_head = nn.Dense(self.vocab_size, use_bias=False)

    def __call__(self, idx, deterministic=True):
        B, T = idx.shape
        tok_emb = self.token_embed(idx)
        pos_emb = self.pos_embed[:, :T, :]
        x = tok_emb + pos_emb
        x = self.dropout_emb(x, deterministic=deterministic)
        for layer in self.layers:
            x = layer(x, deterministic=deterministic)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

In [17]:
def get_batch(rng, split, train_data, val_data, batch_size, block_size):
    data = train_data if split == "train" else val_data
    n = len(data) - block_size
    indices = jax.random.randint(rng, (batch_size,), 0, n)
    x = jnp.stack([data[i:i+block_size] for i in indices])
    y = jnp.stack([data[i+1:i+block_size+1] for i in indices])
    return x, y

def cross_entropy_loss(logits, targets):
    B, T, V = logits.shape
    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)
    return optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()

def compute_loss(params, rng, batch, deterministic):
    # params is the raw parameter dict (without the outer 'params' key)
    logits = model.apply({'params': params}, batch[0], deterministic=deterministic, rngs={"dropout": rng})
    loss = cross_entropy_loss(logits, batch[1])
    return loss, logits

In [18]:
@jax.jit
def train_step(state, rng, batch):
    def loss_fn(params):
        loss, _ = compute_loss(params, rng, batch, deterministic=False)
        return loss
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

@jax.jit
def eval_step(params, rng, batch):
    loss, _ = compute_loss(params, rng, batch, deterministic=True)
    return loss

def estimate_loss(params, rng, train_data, val_data, batch_size, block_size, eval_iters):
    losses = {"train": 0.0, "val": 0.0}
    for split in ["train", "val"]:
        total_loss = 0.0
        for _ in range(eval_iters):
            rng, batch_rng = jax.random.split(rng)
            batch = get_batch(batch_rng, split, train_data, val_data, batch_size, block_size)
            loss = eval_step(params, batch_rng, batch)
            total_loss += loss
        losses[split] = total_loss / eval_iters
    return losses, rng

In [19]:
model = GPT(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    block_size=block_size,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout_rate=dropout_rate,
)

In [20]:

def create_train_state(rng, learning_rate, weight_decay):
    params = model.init(rng, jnp.ones((batch_size, block_size), dtype=jnp.int32))["params"]
    tx = optax.adamw(learning_rate, weight_decay=weight_decay)
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

In [21]:

rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)
state = create_train_state(init_rng, max_lr, weight_decay)

In [22]:

train_data_np = jnp.array(train_data)
val_data_np = jnp.array(val_data)

for step in tqdm(range(total_steps)):
    rng, batch_rng = jax.random.split(rng)
    batch = get_batch(batch_rng, "train", train_data_np, val_data_np, batch_size, block_size)
    state, loss = train_step(state, batch_rng, batch)

    if (step + 1) % eval_iters == 0:
        losses, rng = estimate_loss(state.params, rng, train_data_np, val_data_np,
                                    batch_size, block_size, eval_iters)
        print(f"step {step+1}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

 10%|█         | 100/1000 [03:33<8:23:02, 33.54s/it]

step 100: train loss 2.6409, val loss 2.6448


 20%|██        | 200/1000 [06:55<7:18:36, 32.90s/it]

step 200: train loss 2.5130, val loss 2.5190


 30%|███       | 300/1000 [10:17<6:24:05, 32.92s/it]

...

step 800: train loss 1.8914, val loss 2.0134


 90%|█████████ | 900/1000 [30:25<55:19, 33.19s/it]  

step 900: train loss 1.8020, val loss 1.9457


100%|██████████| 1000/1000 [33:46<00:00,  2.03s/it]

step 1000: train loss 1.7327, val loss 1.8871


In [23]:
def generate(params, prompt, max_new_tokens, rng):
    idx = jnp.array([encode(prompt)], dtype=jnp.int32)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model.apply({"params": params}, idx_cond, deterministic=True)
        logits = logits[0, -1, :]
        next_token = jnp.argmax(logits, axis=-1)
        idx = jnp.concatenate([idx, next_token[None, None]], axis=1)
    return decode(idx[0].tolist())

In [24]:
print(generate(state.params, "ROMEO:\n", 200, rng))

ROMEO:
The shall the shall the sould the so the sould the so the son
The so the some the sould the some the son the son
The so the sould the some the some the son the sto the sto the st
The stree the the sta
